# Avance 3. Baseline

## Equipo 41

**Proyecto:** Detección de ocultamiento de mercancía en video para retail

### Objetivo

Este notebook documenta el desarrollo y evaluación del modelo baseline utilizado como referencia para los experimentos posteriores del proyecto integrador. El objetivo es establecer un punto de comparación reproducible que permita evaluar la viabilidad de distinguir eventos de `ocultamiento` y `sin_ocultamiento` a partir de clips de video.

Debido a que el problema corresponde al reconocimiento de acciones en video, se utiliza una arquitectura R3D-18 implementada en PyTorch y preentrenada sobre Kinetics-400. Esta arquitectura permite procesar directamente secuencias RGB y aprender representaciones espacio-temporales a partir de la información visual y del movimiento presente entre frames. El modelo se adapta al problema mediante transferencia de aprendizaje y ajuste fino para clasificación binaria.

El desarrollo del baseline sigue la fase de modelado de CRISP-ML(Q), considerando tanto el desempeño predictivo como aspectos relacionados con generalización, complejidad, explicabilidad y demanda de recursos. La evaluación se realiza sobre conjuntos separados de entrenamiento, validación y prueba, manteniendo controles para evitar fuga de información entre clips relacionados y conservando la fuente como variable de control para analizar generalización.

Además del desempeño del modelo, este notebook analiza las representaciones aprendidas por R3D-18, el posible subajuste o sobreajuste durante el entrenamiento, las métricas apropiadas para el problema de clasificación binaria y el nivel de desempeño alcanzado respecto a referencias básicas.

### Relación entre la rúbrica y el tratamiento del baseline de video

La rúbrica del Avance 3 está formulada de manera general para problemas de aprendizaje automático. En este proyecto, sus criterios se adaptan a un problema de clasificación binaria de acciones sobre datos de video no estructurados. Esta adaptación conserva el propósito de cada criterio, pero considera que las características utilizadas para la clasificación no corresponden a variables tabulares definidas manualmente, sino a representaciones espacio-temporales aprendidas por una red neuronal convolucional 3D.

| Criterio de la rúbrica | Aplicación en este notebook |
|---|---|
| Algoritmo | Selección y justificación de R3D-18 como modelo baseline para reconocimiento de acciones en video. Se consideran la naturaleza espacio-temporal de los datos, el tamaño disponible del conjunto de entrenamiento, el uso de pesos preentrenados en Kinetics-400, la complejidad del modelo, sus requerimientos computacionales y sus limitaciones de interpretabilidad. |
| Características importantes | Las características no se seleccionan directamente a partir de columnas tabulares. R3D-18 aprende representaciones espacio-temporales durante el procesamiento de los clips. Se extraen los embeddings generados antes de la capa de clasificación y se analiza su estructura mediante reducción de dimensionalidad y representación visual. |
| Subajuste / sobreajuste | Se compara la evolución de la función de pérdida y de las métricas de desempeño entre entrenamiento y validación. También se comparan los resultados obtenidos en entrenamiento, validación y prueba para identificar posibles brechas de generalización. |
| Métrica | La evaluación se adapta a un problema de clasificación binaria con desbalance de clases. Las métricas principales serán ROC-AUC, PR-AUC y EER, dando especial atención a PR-AUC por el comportamiento sobre la clase minoritaria. Se complementan con matriz de confusión, precision, recall, F1-score, specificity y accuracy para analizar el comportamiento a un umbral de decisión determinado. |
| Desempeño | El resultado del modelo se compara contra referencias básicas y contra el comportamiento esperado de un clasificador no informativo. El desempeño obtenido se utiliza como punto de referencia para determinar posteriormente si modelos alternativos proporcionan mejoras relevantes. |

La selección y evaluación del modelo se realiza siguiendo CRISP-ML(Q), por lo que el análisis no se limita a una única métrica predictiva. También se documentan aspectos relacionados con generalización, tiempo de entrenamiento, recursos utilizados, complejidad y capacidad de interpretación de las representaciones aprendidas.

## 1. Definición del problema de modelado

El proyecto se formula como un problema de aprendizaje supervisado para clasificación binaria de acciones en video. El objetivo del modelo es distinguir entre clips que contienen un evento observable de `ocultamiento` de mercancía y clips que representan comportamiento `sin_ocultamiento`.

La unidad de análisis es un clip temporal de video y no el video completo. Esta decisión es necesaria porque un video puede contener diferentes acciones a lo largo del tiempo y un evento de ocultamiento puede representar solamente una fracción de su duración. En consecuencia, la etiqueta asignada a un video no se transfiere automáticamente a todos sus fragmentos.

Para el baseline, cada observación se representa mediante una secuencia RGB de 16 frames con resolución espacial de 112 × 112 píxeles. La entrada esperada por el modelo tiene la forma:

$$
X_i \in \mathbb{R}^{3 \times 16 \times 112 \times 112}
$$

La variable objetivo se define como:

$$
y_i =
\begin{cases}
0, & \text{sin\_ocultamiento} \\
1, & \text{ocultamiento}
\end{cases}
$$

La clase `ocultamiento` corresponde a clips en los que existe evidencia visual suficiente de acercamiento o introducción de mercancía hacia una zona definida de ocultamiento, como bolsillo, ropa, bolsa o mochila. La clase `sin_ocultamiento` incluye comportamientos observables donde la mercancía permanece visible o no se presenta la acción objetivo.

Los clips cuya acción no pueda determinarse con suficiente claridad debido a oclusiones, visibilidad insuficiente, cortes temporales u otras condiciones ambiguas no se fuerzan a una de las dos clases para el entrenamiento principal. Estos casos se conservan, cuando resulte pertinente, para análisis posterior de errores y limitaciones.

El modelo genera dos valores de salida asociados con las clases del problema. Durante el entrenamiento estos valores se utilizan como logits para calcular la función de pérdida y, durante la evaluación, pueden transformarse mediante Softmax en probabilidades estimadas para cada clase.

### 1.1 Decisiones de modelado y criterios de calidad

La definición del baseline considera las características de los datos, las restricciones del proyecto y el propósito de utilizar el modelo como referencia para experimentos posteriores. De acuerdo con CRISP-ML(Q), la selección y evaluación del modelo debe considerar tanto el desempeño predictivo como su capacidad de generalización y las restricciones relevantes para su posible utilización.

| Elemento | Decisión para el baseline |
|---|---|
| Tipo de aprendizaje | Supervisado |
| Tipo de problema | Clasificación binaria de acciones en video |
| Clase positiva | `ocultamiento` |
| Clase negativa | `sin_ocultamiento` |
| Unidad de análisis | Clip temporal |
| Entrada | Video RGB |
| Frames por muestra | 16 |
| Resolución de entrada | 112 × 112 |
| Modelo baseline | R3D-18 |
| Framework | PyTorch / Torchvision |
| Inicialización | Pesos preentrenados en Kinetics-400 |
| Estrategia de entrenamiento | Transfer learning y ajuste fino |
| Salida durante entrenamiento | Dos logits, uno por clase |
| Salida para evaluación | Probabilidades obtenidas mediante Softmax |
| Separación de datos | Entrenamiento, validación y prueba sin compartir entre particiones clips pertenecientes al mismo grupo (dataset, video_id) |
| Tratamiento de casos ambiguos | Exclusión del entrenamiento principal y conservación para análisis cuando sea pertinente |
| Uso previsto | Referencia experimental para comparar modelos posteriores |
| Alcance | Evaluación sobre videos grabados; no corresponde a un sistema productivo |

Además del desempeño predictivo, se consideran los siguientes criterios de calidad:

- **Generalización:** diferencia entre el desempeño obtenido en entrenamiento, validación y prueba.
- **Explicabilidad:** análisis posterior de las representaciones espacio-temporales aprendidas y de los errores de clasificación.
- **Complejidad:** número de parámetros y características de la arquitectura.
- **Demanda de recursos:** tiempo de entrenamiento, memoria utilizada y velocidad de inferencia.
- **Reproducibilidad:** registro de configuración, semillas, particiones, hiperparámetros y versiones de las bibliotecas utilizadas.

La clasificación producida por el modelo se mantiene separada de la lógica operativa de generación de alertas. El baseline estima la probabilidad de cada clase para un clip; la selección posterior de un umbral de decisión permitirá analizar el compromiso entre detecciones correctas y falsas alarmas.

## 2. Protocolo experimental y partición de datos

El Avance 2 estableció las condiciones que deben conservarse durante el
modelado para evitar fuga de información y controlar los principales riesgos
identificados durante Data Preparation.

En particular:

- Únicamente participan los clips elegibles para clasificación binaria;
- Los clips pertenecientes al mismo grupo `(dataset, video_id)` deben
  permanecer dentro de una sola partición;
- La composición de las particiones debe conservar, en la medida de lo
  posible, representación de ambas clases y de las diferentes fuentes;
- Cualquier tratamiento del desbalance se realizará después de la
  partición y exclusivamente sobre entrenamiento;
- La asignación a entrenamiento, validación y prueba se persistirá para
  utilizar exactamente las mismas observaciones al comparar arquitecturas.

A partir de estas condiciones, en este avance se construye y valida la
partición experimental que utilizará el baseline R3D-18.

In [2]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Directorio de trabajo para la preparación de datos
BASE_DIR = Path(
    "/opt/workspace/MNA---Proyecto-Integrador/"
    "Datos/Preparacion_Datasets"
)

# Archivos generados durante la preparación
MANIFEST_PATH = BASE_DIR / "dataset_manifest_todos.csv"

FEATURES_PATH = (
    BASE_DIR / "dataset_consolidado_features.csv"
)

DURACIONES_PATH = (
    BASE_DIR / "duraciones_dataset_consolidado.csv"
)

# Dataset físico consolidado
CLIPS_DIR = Path(
    "/opt/workspace/datasets/procesados/clips_4s"
)

print("Manifest disponible:    ", MANIFEST_PATH.exists())
print("Features disponibles:    ", FEATURES_PATH.exists())
print("Duraciones disponibles:  ", DURACIONES_PATH.exists())
print("Directorio de clips:     ", CLIPS_DIR.exists())

# El archivo de features será la tabla principal para definir
# el protocolo experimental del baseline
if not FEATURES_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo esperado: {FEATURES_PATH}"
    )

df = pd.read_csv(FEATURES_PATH)

print(f"\nRegistros: {len(df):,}")
print(f"Columnas:  {len(df.columns)}")

print("\nColumnas disponibles:")
print(df.columns.tolist())

print("\nPrimeros registros:")
display(df.head())

Manifest disponible:     True
Features disponibles:     True
Duraciones disponibles:   True
Directorio de clips:      True

Registros: 987
Columnas:  19

Columnas disponibles:
['dataset', 'video_id', 'video_origen', 'clip_id', 'inicio_s', 'fin_s', 'etiqueta', 'persona_id', 'evento_id', 'nota', 'nombre_clip', 'duracion_real_s', 'duracion_objetivo_s', 'diferencia_vs_4s', 'cumple_4s', 'requiere_normalizacion_temporal', 'elegible_modelo_binario', 'motivo_no_elegible', 'clase_binaria']

Primeros registros:


,dataset,video_id,video_origen,clip_id,inicio_s,fin_s,etiqueta,persona_id,evento_id,nota,nombre_clip,duracion_real_s,duracion_objetivo_s,diferencia_vs_4s,cumple_4s,requiere_normalizacion_temporal,elegible_modelo_binario,motivo_no_elegible,clase_binaria
0,CCTV_YOLO,CCTV_001,not_shoplifting1.mp4,CCTV_001,2.0,6.0,sin_ocultamiento,P001_AJ,NaN,NaN,CCTV_YOLO__CCTV_001__CCTV_001.mp4,4.0,4.0,0.0,True,False,True,NaN,0.0
1,CCTV_YOLO,CCTV_002,not_shoplifting2.mp4,CCTV_002,2.0,6.0,sin_ocultamiento,P002_AJ,NaN,NaN,CCTV_YOLO__CCTV_002__CCTV_002.mp4,4.0,4.0,0.0,True,False,True,NaN,0.0
2,CCTV_YOLO,CCTV_003,not_shoplifting3.mp4,CCTV_003,2.0,6.0,sin_ocultamiento,P003_AJ,NaN,NaN,CCTV_YOLO__CCTV_003__CCTV_003.mp4,4.0,4.0,0.0,True,False,True,NaN,0.0
3,CCTV_YOLO,CCTV_004,not_shoplifting4.mp4,CCTV_004,2.0,6.0,sin_ocultamiento,P004_AJ,NaN,NaN,CCTV_YOLO__CCTV_004__CCTV_004.mp4,4.0,4.0,0.0,True,False,True,NaN,0.0
4,CCTV_YOLO,CCTV_005,shoplifting1.mp4,CCTV_005,2.0,6.0,ocultamiento,P001_AJ,NaN,NaN,CCTV_YOLO__CCTV_005__CCTV_005.mp4,4.0,4.0,0.0,True,False,True,NaN,1.0


### 2.1 Unidad de agrupación

El Avance 2 definió la combinación `(dataset, video_id)` como unidad mínima
de agrupación para prevenir fuga de información entre clips relacionados.
Esta regla se conserva durante la construcción de las particiones.

Antes de generar el split definitivo se verifica adicionalmente `persona_id`,
ya que el Avance 2 estableció que los identificadores confiables de persona,
evento o escenario deben considerarse cuando permitan detectar dependencia
entre observaciones pertenecientes a videos distintos.

Para el modelado se utilizan únicamente registros con
`elegible_modelo_binario = True` y una clase binaria definida.

In [5]:
# Subconjunto utilizado para el baseline
df_modelo = df[
    df["elegible_modelo_binario"].eq(True)
    & df["clase_binaria"].notna()
].copy()

df_modelo["clase_binaria"] = (
    df_modelo["clase_binaria"].astype(int)
)

df_modelo["grupo_video"] = (
    df_modelo["dataset"].astype(str)
    + "__"
    + df_modelo["video_id"].astype(str)
)

print(f"Clips elegibles: {len(df_modelo):,}")
print(f"Grupos de video: {df_modelo['grupo_video'].nunique():,}")

assert len(df_modelo) == 985
assert set(df_modelo["clase_binaria"].unique()) == {0, 1}

Clips elegibles: 985
Grupos de video: 229


In [7]:
# Verificar cuántas clases aparecen dentro de cada video
clases_por_video = (
    df_modelo
    .groupby("grupo_video")["clase_binaria"]
    .nunique()
)

videos_multiclase = clases_por_video[
    clases_por_video > 1
]

print(
    "Videos que contienen clips de más de una clase:",
    len(videos_multiclase)
)

Videos que contienen clips de más de una clase: 16
